# ECG Heart-Attack Detection — Training & Evaluation on Google Colab

**Task:** given one lead of an ECG (Lead I, 10 seconds, 100 Hz), decide
**Normal (0)** vs **Myocardial Infarction / heart attack (1)**.

**Data:** PTB-XL, already filtered and packed on the laptop into a single file
`ptbxl_leadI.npz` (14,538 recordings: 9,069 Normal, 5,469 MI).

**Before you run this notebook**
1. On your laptop run `python 3_export_for_colab.py` — it makes `data/ptbxl_leadI.npz` (~22 MB).
2. In Google Drive create a folder **`MyDrive/ecg_project/`** and upload that file into it.
3. In Colab: **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

Everything the notebook produces (trained model, metrics, figures) is written back
to `MyDrive/ecg_project/outputs/`, so it survives after the Colab session ends.

## 1. Check the runtime

In [ ]:
# Confirm TensorFlow can see a GPU. Not strictly required (this model is small
# enough for CPU) but training is ~10x faster with one.
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU found:", gpus[0].name)
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else:
    print("No GPU. It will still work, just slower.")
    print("Tip: Runtime -> Change runtime type -> T4 GPU")

## 2. Connect Google Drive

A pop-up will ask you to sign in and allow access. This is how Colab reads the
dataset you uploaded and writes the results back.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

# Where your project lives inside Google Drive.
# Change this ONE line if you put the file somewhere else.
DRIVE_DIR = "/content/drive/MyDrive/ecg_project"

DATA_PATH = os.path.join(DRIVE_DIR, "ptbxl_leadI.npz")
OUT_DIR   = os.path.join(DRIVE_DIR, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}\n"
        "Upload data/ptbxl_leadI.npz into MyDrive/ecg_project/ and re-run this cell."
    )

size_mb = os.path.getsize(DATA_PATH) / (1024 * 1024)
print(f"Found dataset : {DATA_PATH}  ({size_mb:.1f} MB)")
print(f"Results go to : {OUT_DIR}")

## 3. Load the data

In [ ]:
import numpy as np

bundle = np.load(DATA_PATH)
X_raw = bundle["X"]        # (N, 1000)  Lead I, 100 Hz, 10 seconds
y     = bundle["y"]        # 0 = Normal, 1 = MI
folds = bundle["folds"]    # 1..10, the official PTB-XL split numbers

print("Signals :", X_raw.shape, X_raw.dtype)
print("Labels  :", y.shape)
print()
print(f"Normal (0)       : {(y == 0).sum():5d}  ({100 * (y == 0).mean():.1f}%)")
print(f"Heart attack (1) : {(y == 1).sum():5d}  ({100 * (y == 1).mean():.1f}%)")

### A quick look at the signals

Always eyeball the data before trusting a model. Below are three Normal and
three MI recordings, drawn straight from the array we just loaded.

In [ ]:
import matplotlib.pyplot as plt

t = np.arange(X_raw.shape[1]) / 100.0     # seconds (100 samples per second)
rng = np.random.default_rng(0)
normal_ids = rng.choice(np.where(y == 0)[0], 3, replace=False)
mi_ids     = rng.choice(np.where(y == 1)[0], 3, replace=False)

fig, axes = plt.subplots(3, 2, figsize=(13, 6), sharex=True)
for row in range(3):
    axes[row, 0].plot(t, X_raw[normal_ids[row]], lw=0.8, color="#2a7f4f")
    axes[row, 1].plot(t, X_raw[mi_ids[row]],     lw=0.8, color="#b03a3a")
    axes[row, 0].set_ylabel("mV")
axes[0, 0].set_title("Normal")
axes[0, 1].set_title("Myocardial Infarction (MI)")
axes[2, 0].set_xlabel("seconds"); axes[2, 1].set_xlabel("seconds")
plt.tight_layout()
plt.show()

## 4. Prepare the data

Two things happen here.

**Normalise (z-score).** Every recording is rescaled to mean 0, standard
deviation 1. Different machines and different electrode placements produce
signals of different sizes; we want the model to learn the *shape* of the
heartbeat, not how big the numbers happen to be.

**Split by the official folds.** PTB-XL ships a `strat_fold` number (1–10) for
every recording, built so that **the same patient never appears in two folds**.
Using it means our test score is honest — the model cannot have memorised a
patient it will be examined on.

| Folds | Used as | Why |
|---|---|---|
| 1–8 | Training | The model learns from these. |
| 9 | Validation | Picks the stopping point and the decision threshold. |
| 10 | Test | Touched **once**, at the very end. The real exam. |

In [ ]:
# --- normalise each recording on its own ---
mean = X_raw.mean(axis=1, keepdims=True)
std  = X_raw.std(axis=1, keepdims=True) + 1e-8      # +tiny: never divide by zero
X = (X_raw - mean) / std

# Conv1D expects a "channel" dimension: (N, 1000) -> (N, 1000, 1)
X = X[..., np.newaxis].astype(np.float32)

# --- split using the official fold numbers ---
train_mask = np.isin(folds, [1, 2, 3, 4, 5, 6, 7, 8])
val_mask   = folds == 9
test_mask  = folds == 10

X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

for name, yy in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"{name:6s}: {len(yy):5d} recordings   "
          f"Normal {int((yy == 0).sum()):4d}   MI {int((yy == 1).sum()):4d}   "
          f"({100 * yy.mean():.1f}% MI)")

### Class weights

There are roughly 1.7 Normal recordings for every MI recording. Left alone, a
lazy model could score 62% just by always answering "Normal". Class weights make
each MI mistake cost more, so that shortcut stops paying off.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
class_weight = {0: float(weights[0]), 1: float(weights[1])}
print("Class weights:", {k: round(v, 3) for k, v in class_weight.items()})

## 5. Build the model

A **1D Convolutional Neural Network**. A 2D CNN slides small filters over an
image; a 1D CNN slides them along a signal, learning to recognise shapes such as
a QRS complex or a depressed ST segment wherever they occur in the 10 seconds.

Each block does the same four things:

- **Conv1D** — slide filters along the signal to find patterns.
- **BatchNormalization** — keep the numbers in a healthy range so training is stable.
- **ReLU** — lets the network learn curved, non-straight-line relationships.
- **MaxPooling1D** — halve the length, keeping the strongest responses.

Stacking four blocks means later layers see a wider stretch of time at once:
block 1 looks at fractions of a second, block 4 at whole beats.

At the end we pool over time in two ways — the **average** response (was this
pattern present overall?) and the **maximum** (did it appear strongly anywhere?)
— then a small dense layer turns that summary into one number between 0 and 1:
the estimated probability of MI.

In [ ]:
from tensorflow.keras import layers, models

# Wipe any model left over from a previous run of this cell. Without this,
# re-running would rename the AUC metric to "auc_1" and the callbacks below,
# which watch "val_auc", would quietly stop working.
tf.keras.backend.clear_session()

# Same seed every run so your thesis numbers are reproducible.
SEED = 42
tf.keras.utils.set_random_seed(SEED)


def conv_block(x, filters, kernel_size, dropout):
    # Two convolutions per block: the second one refines what the first found.
    for _ in range(2):
        x = layers.Conv1D(filters, kernel_size, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(dropout)(x)
    return x


def build_model(input_length):
    inp = layers.Input(shape=(input_length, 1))

    x = conv_block(inp, 32,  7, 0.10)     # 1000 -> 500 time steps
    x = conv_block(x,   64,  5, 0.10)     #  500 -> 250
    x = conv_block(x,  128,  3, 0.20)     #  250 -> 125
    x = conv_block(x,  128,  3, 0.20)     #  125 ->  62

    # Summarise the whole signal two different ways, then join them.
    avg = layers.GlobalAveragePooling1D()(x)
    mx  = layers.GlobalMaxPooling1D()(x)
    x = layers.Concatenate()([avg, mx])

    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.5)(x)            # switch off half the neurons while
                                          # training -> less over-fitting
    out = layers.Dense(1, activation="sigmoid")(x)   # probability of MI

    model = models.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return model


model = build_model(X.shape[1])
model.summary()

## 6. Train

Two helpers run automatically while training:

- **ReduceLROnPlateau** — when validation stops improving, take smaller learning
  steps instead of giving up.
- **EarlyStopping** — if validation AUC has not improved for 10 epochs, stop and
  restore the best weights seen. This is what prevents over-fitting, and it is
  why we need a validation fold that is separate from the test fold.

On a T4 GPU this takes roughly 3–6 minutes.

In [ ]:
from tensorflow.keras import callbacks

MODEL_PATH = os.path.join(OUT_DIR, "mi_cnn_model.keras")

cbs = [
    callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=10,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5,
                                patience=4, min_lr=1e-5, verbose=1),
    callbacks.ModelCheckpoint(MODEL_PATH, monitor="val_auc", mode="max",
                              save_best_only=True, verbose=0),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=64,
    class_weight=class_weight,
    callbacks=cbs,
    verbose=1,
)

print("\nBest model saved to:", MODEL_PATH)

In [ ]:
# Training curves - loss, accuracy and AUC over time.
h = history.history
epochs = range(1, len(h["loss"]) + 1)

# Keras occasionally suffixes a metric name (auc -> auc_1) if you re-run the
# build cell in the same session, so look the key up instead of hard-coding it.
auc_key = next(k for k in h if k.startswith("auc"))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, title in [(axes[0], "loss", "Loss"),
                       (axes[1], "accuracy", "Accuracy"),
                       (axes[2], auc_key, "ROC-AUC")]:
    ax.plot(epochs, h[key], label="train")
    ax.plot(epochs, h["val_" + key], label="validation")
    ax.set_title(f"{title} over epochs")
    ax.set_xlabel("epoch"); ax.set_ylabel(title.lower()); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

# A large gap between the two lines means over-fitting; they should track together.

## 7. Evaluate

### 7.1 Choosing the decision threshold

The model outputs a probability. Turning it into a yes/no answer needs a cut-off.
The default 0.5 is arbitrary, and it is the wrong default for medicine: **missing
a real heart attack is far worse than a false alarm**.

So we pick the threshold on the **validation fold** (fold 9) and then apply it,
unchanged, to the test fold. Choosing it on the test fold would be cheating —
the score would no longer reflect unseen data.

Two candidates are reported:
- **Best-F1** — the balanced choice.
- **High-recall** — the lowest threshold that still catches at least 90% of real
  MIs, which is closer to how a screening tool would actually be set up.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

val_probs  = model.predict(X_val,  verbose=0).ravel()
test_probs = model.predict(X_test, verbose=0).ravel()

# --- threshold that maximises F1 on validation ---
prec, rec, thr = precision_recall_curve(y_val, val_probs)
# precision_recall_curve returns one more point than thresholds; drop the last.
f1_curve = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
thr_f1 = float(thr[np.argmax(f1_curve)])

# --- lowest threshold reaching >= 90% recall on validation ---
TARGET_RECALL = 0.90
ok = rec[:-1] >= TARGET_RECALL
thr_recall = float(thr[ok][np.argmax(prec[:-1][ok])]) if ok.any() else 0.5

print(f"Threshold (default)      : 0.500")
print(f"Threshold (best F1)      : {thr_f1:.3f}   -> val F1 {f1_curve.max():.3f}")
print(f"Threshold (recall >= 90%): {thr_recall:.3f}")

### 7.2 Test-set results

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, classification_report)
import pandas as pd


def evaluate(y_true, probs, threshold):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    return {
        "Threshold":   round(float(threshold), 3),
        "Accuracy":    accuracy_score(y_true, preds),
        "Precision":   precision_score(y_true, preds, zero_division=0),
        "Recall":      recall_score(y_true, preds),          # = sensitivity
        "Specificity": tn / (tn + fp),
        "F1":          f1_score(y_true, preds),
        "ROC-AUC":     roc_auc_score(y_true, probs),         # threshold-free
        "PR-AUC":      average_precision_score(y_true, probs),
        "Missed MI":   int(fn),
        "False alarm": int(fp),
    }


rows = {
    "Default (0.50)":            evaluate(y_test, test_probs, 0.5),
    "Best-F1 (from val)":        evaluate(y_test, test_probs, thr_f1),
    "High-recall (from val)":    evaluate(y_test, test_probs, thr_recall),
}
results = pd.DataFrame(rows).T
display(results.style.format({c: "{:.4f}" for c in
        ["Accuracy", "Precision", "Recall", "Specificity", "F1", "ROC-AUC", "PR-AUC"]}))

In [ ]:
# The single most important number in this project: recall on the test fold.
main = rows["Best-F1 (from val)"]
print("=" * 62)
print("           TEST-SET SUMMARY  (fold 10, never seen in training)")
print("=" * 62)
print(f"  Recall / Sensitivity : {main['Recall']:.4f}   <-- of real heart attacks, how many we caught")
print(f"  Precision            : {main['Precision']:.4f}   of the MI alarms, how many were right")
print(f"  Specificity          : {main['Specificity']:.4f}   of the healthy people, how many we left alone")
print(f"  F1-score             : {main['F1']:.4f}")
print(f"  Accuracy             : {main['Accuracy']:.4f}")
print(f"  ROC-AUC              : {main['ROC-AUC']:.4f}   how well the two classes separate overall")
print(f"  PR-AUC               : {main['PR-AUC']:.4f}   the same idea, but fairer when classes are unbalanced")
print()
print(f"  Missed heart attacks : {main['Missed MI']}")
print(f"  False alarms         : {main['False alarm']}")
print("=" * 62)
print()
print(classification_report(y_test, (test_probs >= thr_f1).astype(int),
                            target_names=["Normal", "MI"], digits=4))

### 7.3 Confusion matrices

In [ ]:
def plot_cm(ax, y_true, probs, threshold, title):
    cm = confusion_matrix(y_true, (probs >= threshold).astype(int), labels=[0, 1])
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1], ["Normal", "MI"]); ax.set_yticks([0, 1], ["Normal", "MI"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title)
    vmax = cm.max()
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}\n({100 * cm[i, j] / cm[i].sum():.1f}%)",
                    ha="center", va="center", fontsize=12,
                    color="white" if cm[i, j] > vmax / 2 else "black")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
plot_cm(axes[0], y_test, test_probs, 0.5,        "Default threshold 0.50")
plot_cm(axes[1], y_test, test_probs, thr_f1,     f"Best-F1 threshold {thr_f1:.2f}")
plot_cm(axes[2], y_test, test_probs, thr_recall, f"High-recall threshold {thr_recall:.2f}")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()

# Bottom-left cell = missed heart attacks. That is the one to keep small.

### 7.4 ROC and Precision–Recall curves

These show performance at *every* possible threshold, so they do not depend on
the cut-off we happened to pick. The dashed line is what random guessing looks
like.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_probs)
p_curve, r_curve, _ = precision_recall_curve(y_test, test_probs)
roc_auc = roc_auc_score(y_test, test_probs)
pr_auc  = average_precision_score(y_test, test_probs)
baseline = y_test.mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, lw=2, label=f"CNN (AUC = {roc_auc:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="random guessing")
axes[0].set_xlabel("False positive rate  (1 - specificity)")
axes[0].set_ylabel("True positive rate  (recall)")
axes[0].set_title("ROC curve - test fold")

axes[1].plot(r_curve, p_curve, lw=2, color="#b03a3a", label=f"CNN (AP = {pr_auc:.3f})")
axes[1].axhline(baseline, ls="--", c="k", lw=1, label=f"random guessing ({baseline:.2f})")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall curve - test fold")

for ax in axes:
    ax.legend(loc="lower left"); ax.grid(alpha=0.3); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "roc_pr_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

### 7.5 How certain are these numbers?

The test fold holds about 1,460 recordings, so every score carries some sampling
noise. Bootstrapping — re-sampling the test set with replacement 1,000 times and
re-scoring each time — turns each number into a **95% confidence interval**.

Reporting `0.87 (0.85–0.89)` instead of a bare `0.87` is the honest way to write
this up, and it tells you which differences between models are real and which are
noise.

In [ ]:
def bootstrap_ci(y_true, probs, threshold, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    acc, out = {"Recall": [], "Precision": [], "F1": [], "ROC-AUC": [], "Accuracy": []}, {}
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt, pb = y_true[idx], probs[idx]
        if yt.sum() == 0 or yt.sum() == len(yt):
            continue                       # a resample with only one class: skip
        pr = (pb >= threshold).astype(int)
        acc["Recall"].append(recall_score(yt, pr))
        acc["Precision"].append(precision_score(yt, pr, zero_division=0))
        acc["F1"].append(f1_score(yt, pr))
        acc["ROC-AUC"].append(roc_auc_score(yt, pb))
        acc["Accuracy"].append(accuracy_score(yt, pr))
    for k, v in acc.items():
        lo, hi = np.percentile(v, [2.5, 97.5])
        out[k] = (float(np.mean(v)), float(lo), float(hi))
    return out


ci = bootstrap_ci(y_test, test_probs, thr_f1)
print(f"95% confidence intervals on the test fold (1,000 bootstrap resamples,")
print(f"threshold = {thr_f1:.3f} chosen on validation)\n")
for metric, (m, lo, hi) in ci.items():
    print(f"  {metric:10s} {m:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")

## 8. Save everything back to Drive

The Colab machine is deleted when the session ends. This cell writes the model,
the metrics and the figures into `MyDrive/ecg_project/outputs/`, where they stay.

In [ ]:
import json

# The trained model was already checkpointed to Drive during training.
# Save the preprocessing constants the Flask app will need to reproduce
# exactly what the model was trained on.
config = {
    "task": "PTB-XL Lead I, Normal (0) vs Myocardial Infarction (1)",
    "sample_rate_hz": 100,
    "input_length": int(X.shape[1]),
    "preprocessing": "per-recording z-score: (x - mean) / (std + 1e-8)",
    "threshold_default": 0.5,
    "threshold_best_f1": round(thr_f1, 4),
    "threshold_high_recall": round(thr_recall, 4),
    "split": {"train": "folds 1-8", "val": "fold 9", "test": "fold 10"},
    "n_train": int(len(y_train)), "n_val": int(len(y_val)), "n_test": int(len(y_test)),
    "epochs_run": len(history.history["loss"]),
    "seed": SEED,
}
with open(os.path.join(OUT_DIR, "model_config.json"), "w") as f:
    json.dump(config, f, indent=2)

# Full metrics table, both as JSON (for code) and text (for the report).
metrics = {
    "test_results": {k: {kk: (float(vv) if isinstance(vv, (int, float, np.floating)) else vv)
                         for kk, vv in v.items()} for k, v in rows.items()},
    "bootstrap_95ci_at_best_f1": {k: {"mean": v[0], "lo": v[1], "hi": v[2]}
                                  for k, v in ci.items()},
}
with open(os.path.join(OUT_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

results.to_csv(os.path.join(OUT_DIR, "metrics_table.csv"))

with open(os.path.join(OUT_DIR, "metrics.txt"), "w") as f:
    f.write("TEST-SET RESULTS (PTB-XL fold 10, Lead I, Normal vs MI)\n")
    f.write("=" * 62 + "\n\n")
    f.write(results.to_string() + "\n\n")
    f.write("95% confidence intervals (1000 bootstrap resamples, "
            f"threshold={thr_f1:.3f}):\n")
    for metric, (m, lo, hi) in ci.items():
        f.write(f"  {metric:10s} {m:.4f}  [{lo:.4f}, {hi:.4f}]\n")
    f.write("\n" + classification_report(y_test, (test_probs >= thr_f1).astype(int),
                                         target_names=["Normal", "MI"], digits=4))

# Raw test predictions - lets you redo any analysis without retraining.
np.savez_compressed(os.path.join(OUT_DIR, "test_predictions.npz"),
                    y_true=y_test, probs=test_probs)

print("Saved to", OUT_DIR)
for fname in sorted(os.listdir(OUT_DIR)):
    mb = os.path.getsize(os.path.join(OUT_DIR, fname)) / (1024 * 1024)
    print(f"  {fname:26s} {mb:6.2f} MB")

### Download the model to your laptop (optional)

The Flask web app needs `mi_cnn_model.keras` and `model_config.json`. You can
either grab them from Google Drive directly, or run the cell below.

In [ ]:
from google.colab import files

files.download(os.path.join(OUT_DIR, "mi_cnn_model.keras"))
files.download(os.path.join(OUT_DIR, "model_config.json"))

---

## What to write in the thesis

- **Data:** PTB-XL, 21,799 recordings filtered to 14,538 that are cleanly Normal
  (9,069) or MI (5,469). Lead I only, 100 Hz, 10 seconds.
- **Why Lead I:** it is the lead a cheap two-electrode device (ESP32 + AD8232)
  can realistically capture, so the model matches the hardware.
- **Split:** the official PTB-XL `strat_fold` — patient-disjoint, so no leakage.
  Folds 1–8 train, 9 validation, 10 test.
- **Model:** four-block 1D CNN, batch normalisation, dropout, ~few hundred
  thousand parameters.
- **Imbalance:** handled with class weights, not resampling.
- **Threshold:** tuned on validation, never on test — report both the best-F1 and
  the high-recall operating points.
- **Reported metrics:** recall (primary — missing an MI is the costly error),
  precision, specificity, F1, ROC-AUC, PR-AUC, each with a bootstrap 95% CI.

**Known limitations to state honestly**
- Trained on clean hospital recordings; a real ESP32 signal is noisier, so
  performance on live hardware will be lower.
- Single lead loses information a full 12-lead ECG carries — the standard clinical
  criteria for MI localisation need the other leads.
- Not a medical device and not validated for clinical use.